# Restaurant Annual Turnover Prediction

This notebook builds a **regression model** to predict `Annual Turnover` for restaurants using the hackathon training and test datasets.

We will go through:
- Data loading
- Exploratory Data Analysis (EDA)
- Missing value treatment
- Feature engineering & encoding
- Model training & evaluation
- Prediction on test data
- Submission file creation


In [ ]:
## Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


## 1. Load the data

In [ ]:
train_path = 'Train dataset.csv'
test_path = 'Test dataset.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

df_train.head()

In [ ]:
df_train.info()

In [ ]:
df_train.isnull().sum()

From the original sample code, we know there are missing values in columns like:
- `Facebook Popularity Quotient`
- `Instagram Popularity Quotient`
- Several rating columns (e.g., `Live Music Rating`, `Comedy Gigs Rating`, `Value Deals Rating`, `Live Sports Rating`, etc.)

We will handle missing values separately for numeric and categorical features.

## 2. Basic EDA

In [ ]:
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_cols, categorical_cols

In [ ]:
df_train[numeric_cols].describe().T

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_train[' Annual Turnover'], kde=True)
plt.title('Distribution of Annual Turnover')
plt.show()

In [ ]:
corr = df_train[numeric_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

## 3. Feature selection & preprocessing

From the original hackathon sample code, we know they used:
- `Instagram Popularity Quotient` as a key predictor
- `Annual Turnover` as the target

We will build a richer model using multiple features, but still include `Instagram Popularity Quotient` explicitly.

We will:
- Treat **numeric features** with mean imputation
- Treat **categorical features** with most-frequent imputation + one-hot encoding


In [ ]:
target_col = ' Annual Turnover'

# Drop obvious identifiers from features
id_cols = ['Registration Number']

feature_cols = [c for c in df_train.columns if c not in [target_col] + id_cols]

X = df_train[feature_cols]
y = df_train[target_col]

X_test_full = df_test[feature_cols]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_features, categorical_features

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


## 4. Train–test split and model training

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)

y_pred = model.predict(X_valid)

rmse = mean_squared_error(y_valid, y_pred, squared=False)
r2 = r2_score(y_valid, y_pred)

rmse, r2

## 5. Error analysis

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_valid, y_pred, alpha=0.6)
plt.xlabel('Actual Annual Turnover')
plt.ylabel('Predicted Annual Turnover')
plt.title('Actual vs Predicted')
plt.plot([y_valid.min(), y_valid.max()], [y_valid.min(), y_valid.max()], 'r--')
plt.show()

In [ ]:
residuals = y_valid - y_pred
plt.figure(figsize=(8, 4))
sns.histplot(residuals, kde=True)
plt.title('Residuals Distribution')
plt.show()

## 6. Simple model using only Instagram Popularity Quotient (to mirror sample code)

The original hackathon sample code specifically showed:

```python
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

SI = SimpleImputer(strategy='mean')
SI.fit(df_train[[' Instagram Popularity Quotient']])
IPQ = SI.transform(df_train[[' Instagram Popularity Quotient']])

LR = LinearRegression()
model_simple = LR.fit(IPQ, df_train[' Annual Turnover'])
model_simple.score(IPQ, df_train[' Annual Turnover'])
```

We reproduce that here as a baseline.

In [ ]:
SI = SimpleImputer(strategy='mean')
IPQ_train = SI.fit_transform(df_train[[' Instagram Popularity Quotient']])

LR_simple = LinearRegression()
model_simple = LR_simple.fit(IPQ_train, df_train[' Annual Turnover'])

simple_score = model_simple.score(IPQ_train, df_train[' Annual Turnover'])
simple_score

## 7. Train final model on full training data

We now retrain the **full feature model** on all available training data and then predict on the test set.

In [ ]:
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

final_model.fit(X, y)

test_predictions = final_model.predict(X_test_full)
test_predictions[:5]

## 8. Create submission file

The original sample code shows building a solution dataframe using `Registration Number` and predicted `Annual Turnover` and then saving it as `Submission.csv`.

In [ ]:
solution_df = pd.DataFrame()
solution_df['Registration Number'] = df_test['Registration Number']
solution_df[' Annual Turnover'] = test_predictions

solution_df.head()

In [ ]:
solution_df.to_csv('Submission.csv', index=False)
print('Submission.csv has been created.')

# End of Notebook

You can now submit `Submission.csv` to the hackathon platform or use it for further analysis.